# New Section

In [6]:
import os
import time
import torch
import gradio as gr
import numpy as np
import psutil
import cv2
from PIL import Image
from diffusers import StableDiffusionPipeline
from skimage.restoration import estimate_sigma


# CLIP (open-clip)
try:
    import open_clip
except:
    import open_clip_torch as open_clip
# BRISQUE (optional)
try:
    from brisque import BRISQUE
    brisque_available = True
except:
    brisque_available = False


# ========== SETUP ==========
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

os.makedirs("outputs", exist_ok=True)

# ---------- Load Stable Diffusion ----------
load_start = time.time()

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

pipe.enable_attention_slicing()
try:
    pipe.enable_xformers_memory_efficient_attention()
except:
    pass

load_end = time.time()
MODEL_LOAD_TIME = round(load_end - load_start, 2)
print("Model loaded in:", MODEL_LOAD_TIME, "seconds")

# ---------- Load CLIP (open-clip) ----------
clip_model, clip_preprocess, clip_tokenizer = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
clip_model = clip_model.to(device)
clip_model.eval()


# ================== METRIC FUNCTIONS ==================

def compute_clip_similarity(pil_img, text):
    img = clip_preprocess(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        image_emb = clip_model.encode_image(img)
        text_emb = clip_model.encode_text(clip_tokenizer(text).to(device))

        image_emb /= image_emb.norm(dim=-1, keepdim=True)
        text_emb /= text_emb.norm(dim=-1, keepdim=True)

        sim = (image_emb @ text_emb.T).cpu().item()

    return ((sim + 1) / 2) * 100   # convert [-1,1] to 0–100%


def sharpness_score(pil_img):
    img = np.array(pil_img.convert("L"))
    var = cv2.Laplacian(img, cv2.CV_64F).var()
    score = min(100, (var / 900) * 100)
    return round(score, 2)


def noise_score(pil_img):
    img = np.array(pil_img.convert("L")) / 255.0
    try:
        sigma = estimate_sigma(img, multichannel=False)
    except:
        sigma = np.std(img - cv2.GaussianBlur(img, (3,3), 0))

    score = (1 - min(sigma, 0.2) / 0.2) * 100
    return round(score, 2), round(float(sigma), 4)


def resolution_quality_pct(pil_img):
    w, h = pil_img.size
    base = 512 * 512
    return min(100, (w * h) / base * 100)


def brisque_score(pil_img, path):
    if not brisque_available:
        return None

    try:
        sc = BRISQUE().score(path)
        quality = 100 - sc   # Lower BRISQUE = higher quality
        return max(0, min(100, quality)), sc
    except:
        return None


def combined_quality(clip_pct, sharp_pct, noise_pct, brisque_pct, res_pct):
    weights = {
        "clip": 0.35,
        "sharp": 0.20,
        "noise": 0.20,
        "brisque": 0.15,
        "res": 0.10
    }

    if brisque_pct is None:
        weights["clip"] += 0.10
        weights["sharp"] += 0.05
        weights["noise"] += 0.05
        weights["brisque"] = 0

    total = (
        clip_pct * weights["clip"]
        + sharp_pct * weights["sharp"]
        + noise_pct * weights["noise"]
        + (brisque_pct or 0) * weights["brisque"]
        + res_pct * weights["res"]
    )

    return round(total, 2)


# ================== MAIN GENERATE FUNCTION ==================

from torch.cuda.amp import autocast

def generate_image_metrics(prompt):
    if not prompt.strip():
        return None, "⚠ Enter a valid prompt."

    start = time.time()

    # Generate
    if device == "cuda":
        with autocast(device_type="cuda"):
            out = pipe(prompt, guidance_scale=7.0)
    else:
        out = pipe(prompt, guidance_scale=7.0)

    img = out.images[0].convert("RGB")
    gen_time = round(time.time() - start, 2)
    path = f"outputs/generated.png"
    img.save(path)

    # Metrics
    clip_pct = compute_clip_similarity(img, prompt)
    sharp_pct = sharpness_score(img)
    noise_pct, sigma = noise_score(img)
    res_pct = resolution_quality_pct(img)

    brisque_result = brisque_score(img, path)
    if brisque_result:
        brisque_pct, raw_brisque = brisque_result
    else:
        brisque_pct = None
        raw_brisque = None

    combined = combined_quality(clip_pct, sharp_pct, noise_pct, brisque_pct, res_pct)

    # Status message
    md = f"""
## 📊 Image Quality Report
**Prompt:** {prompt}

### ⚙ Performance
- Model load time: {MODEL_LOAD_TIME}s
- Generation time: {gen_time}s

### 🖼 Image Quality Metrics (0–100%)
- CLIP similarity: **{clip_pct}%**
- Sharpness: **{sharp_pct}%**
- Noise quality: **{noise_pct}%** (sigma={sigma})
- Resolution score: **{res_pct}%**
"""

    if brisque_pct:
        md += f"- BRISQUE quality: **{brisque_pct}%** (raw={raw_brisque})\n"
    else:
        md += "- BRISQUE not available (optional)\n"

    md += f"\n### ⭐ Final Combined Score: **{combined}%**"

    return img, md


# ================== GRADIO UI ==================

with gr.Blocks() as demo:
    gr.Markdown("## 🚀 Stable Diffusion – Image Generator + Quality Metrics")

    prompt = gr.Textbox(label="Enter Prompt")
    btn = gr.Button("Generate")

    output_img = gr.Image()
    output_md = gr.Markdown()

    btn.click(generate_image_metrics, inputs=prompt, outputs=[output_img, output_md])

demo.launch(share=True)


Running on: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in: 82.15 seconds


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://722db94ee77dfa6402.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# New Section

In [5]:
!pip install git+https://github.com/mlfoundations/open_clip.git

import open_clip

print("OpenCLIP loaded:", open_clip.__version__)


  Cloning https://github.com/mlfoundations/open_clip.git to /tmp/pip-req-build-op32a3m1
  Running command git clone --filter=blob:none --quiet https://github.com/mlfoundations/open_clip.git /tmp/pip-req-build-op32a3m1
  Resolved https://github.com/mlfoundations/open_clip.git to commit d3cdb734a2710feeb4c6307df037afa5f786a3e1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
  Created wheel for open_clip_torch: filename=open_clip_torch-3.2.0-py3-none-any.whl size=1546957 sha256=2e87d13c7c6c3c41e80e3d45a7f371d8fdfc0cf767118f6307388053c1cb382d
  Stored in directory: /tmp/pip-ephem-wheel-cache-3zt4zrwd/wheels/b4/c2/33/0e591fedc30bccff3cead0e2b38fd27aab49a1e543aa9a4360
Successfully built open_clip_torch
OpenCLIP loaded: 3.2.0


# New Section

In [7]:
import torch
from diffusers import StableDiffusionPipeline
import gradio as gr
import os

# Create output folder
os.makedirs("outputs", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("🚀 Running on:", device)

# Load SD v1.5
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

pipe.enable_attention_slicing()

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("✅ xFormers Enabled")
except:
    print("ℹ xFormers not available")

from torch.cuda.amp import autocast   # <-- SAFE + NO WARNINGS

def generate_image(prompt: str):
    if not prompt.strip():
        return None, "⚠ Please enter a prompt."

    if device == "cuda":
        with autocast():
            image = pipe(prompt, guidance_scale=7.0).images[0]
    else:
        image = pipe(prompt, guidance_scale=7.0).images[0]

    img_path = "outputs/generated.png"
    image.save(img_path)

    return image, f"Image generated for: {prompt}"

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 Simple AI Image Generator (Stable Diffusion v1.5)")
    prompt = gr.Textbox(label="Enter your prompt here:")
    generate_btn = gr.Button("Generate Image")

    output_image = gr.Image(label="Generated Image")
    output_text = gr.Textbox(label="Status Message")

    generate_btn.click(generate_image, inputs=prompt, outputs=[output_image, output_text])

if __name__ == "__main__":
    demo.launch()


🚀 Running on: cuda


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

ℹ xFormers not available
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4db982b858a38d81e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
